## 1. 데이터 전처리

In [1]:
%pip install "torch==2.8.0"
%pip install "torchvision==0.23.0" 
%pip install "torchaudio==2.8.0"
%pip install "transformers==4.55.2"
%pip install "tokenizers==0.21.4"
%pip install "safetensors==0.6.2"
%pip install "huggingface-hub==0.34.4"
%pip install "datasets==4.0.0"
%pip install "accelerate==1.10.0"
%pip install "trl==0.21.0"
%pip install "peft==0.17.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.1/888.1 MB 108.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 133.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 167.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 168.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 121.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 112.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 129.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 103.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 113.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 117.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
%pip install hf_transfer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 63.3 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install --upgrade typing_extensions


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
from datasets import load_dataset, Dataset
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

빠른 학습을 위해 학습 데이터와 테스트 데이터를 2:8 비율로 분할합니다. 이 값을 변경하고자 하는 분은 test_ratio의 값을 변경하세요.

In [2]:
from datasets import load_dataset, Dataset
import random

# 시드 고정
SEED = 42
random.seed(SEED)

# 1. 허깅페이스 허브에서 데이터셋 로드
dataset = load_dataset("iamjoon/manufacturing-text-to-sql", split="train")
print(f"원본 데이터 개수: {len(dataset)}")

# 2. train/test split
test_ratio = 0.2
indices = list(range(len(dataset)))
random.shuffle(indices)
test_size = int(len(indices) * test_ratio)
test_indices = indices[:test_size]
train_indices = indices[test_size:]

# 3. OpenAI format 변환 함수
def format_data(sample):
    return {
        "messages": [
            {"role": "system", "content": sample["system_prompt"]},
            {"role": "user", "content": sample["user_prompt"]},
            {"role": "assistant", "content": sample["assistant"]},
        ]
    }

# 4. 변환
train_dataset = [format_data(dataset[i]) for i in train_indices]
test_dataset = [format_data(dataset[i]) for i in test_indices]

# 5. 결과 출력
print(f"Train: {len(train_dataset)}개")
print(f"Test:  {len(test_dataset)}개")

README.md:   0%|          | 0.00/358 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/85.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/776 [00:00<?, ? examples/s]

원본 데이터 개수: 776
Train: 621개
Test:  155개


In [3]:
train_dataset[345]["messages"]

[{'role': 'system',
  'content': "\n당신은 SQL을 생성하는 AI 모델입니다.\n아래는 데이터베이스 스키마(DDL)입니다.\n\n<SCHEMA>\n-- 작업완료로그 테이블\nCREATE TABLE IF NOT EXISTS `log_lottranslog` (\n  `TRANSLOGID` bigint(20) NOT NULL DEFAULT 0 COMMENT '로그ID',\n  `LOTNO` char(20) NOT NULL DEFAULT '' COMMENT 'LOT번호',\n  `LINENO` char(20) DEFAULT NULL COMMENT '생산라인번호',\n  `TRANSACTIONNAME` char(50) NOT NULL DEFAULT '' COMMENT '처리명',\n  `TIMELOGGED` datetime(3) NOT NULL DEFAULT '0000-00-00 00:00:00.000' COMMENT '로그입력일시',\n  `ACTUALTIME` datetime(3) NOT NULL DEFAULT '0000-00-00 00:00:00.000' COMMENT '실제실행일시',\n  `MATERIALCODE` char(30) NOT NULL DEFAULT '' COMMENT '자재코드',\n  `MATERIALNAME` char(50) NOT NULL DEFAULT '' COMMENT '자재명',\n  `TRANSQTY` double NOT NULL DEFAULT 0 COMMENT '변경수량',\n  `CURRENTQTY` double NOT NULL DEFAULT 0 COMMENT '현재수량',\n  `NEXTQTY` double DEFAULT NULL COMMENT '변경반영된수량',\n  `TRANSUOM` char(5) DEFAULT NULL COMMENT '측정단위',\n  `WAREHOUSECODE` char(20) DEFAULT NULL COMMENT '창고코드',\n  `BOPMATERIALCODE` char(3

In [4]:
# 리스트 형태에서 다시 Dataset 객체로 변경
print(type(train_dataset))
print(type(test_dataset))
train_dataset = Dataset.from_list(train_dataset)
test_dataset = Dataset.from_list(test_dataset)
print(type(train_dataset))
print(type(test_dataset))

<class 'list'>
<class 'list'>
<class 'datasets.arrow_dataset.Dataset'>
<class 'datasets.arrow_dataset.Dataset'>


In [5]:
train_dataset[0]

{'messages': [{'content': "\n당신은 SQL을 생성하는 AI 모델입니다.\n아래는 데이터베이스 스키마(DDL)입니다.\n\n<SCHEMA>\n-- 작업완료로그 테이블\nCREATE TABLE IF NOT EXISTS `log_lottranslog` (\n  `TRANSLOGID` bigint(20) NOT NULL DEFAULT 0 COMMENT '로그ID',\n  `LOTNO` char(20) NOT NULL DEFAULT '' COMMENT 'LOT번호',\n  `LINENO` char(20) DEFAULT NULL COMMENT '생산라인번호',\n  `TRANSACTIONNAME` char(50) NOT NULL DEFAULT '' COMMENT '처리명',\n  `TIMELOGGED` datetime(3) NOT NULL DEFAULT '0000-00-00 00:00:00.000' COMMENT '로그입력일시',\n  `ACTUALTIME` datetime(3) NOT NULL DEFAULT '0000-00-00 00:00:00.000' COMMENT '실제실행일시',\n  `MATERIALCODE` char(30) NOT NULL DEFAULT '' COMMENT '자재코드',\n  `MATERIALNAME` char(50) NOT NULL DEFAULT '' COMMENT '자재명',\n  `TRANSQTY` double NOT NULL DEFAULT 0 COMMENT '변경수량',\n  `CURRENTQTY` double NOT NULL DEFAULT 0 COMMENT '현재수량',\n  `NEXTQTY` double DEFAULT NULL COMMENT '변경반영된수량',\n  `TRANSUOM` char(5) DEFAULT NULL COMMENT '측정단위',\n  `WAREHOUSECODE` char(20) DEFAULT NULL COMMENT '창고코드',\n  `BOPMATERIALCODE` char(30) DEFA

## 2. 모델 로드 및 템플릿 적용

In [6]:
# 허깅페이스 모델 ID
model_id = "NCSOFT/Llama-VARCO-8B-Instruct" 

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

In [7]:
# 템플릿 적용
text = tokenizer.apply_chat_template(
    train_dataset[0]["messages"], tokenize=False, add_generation_prompt=False
)
print(text)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

당신은 SQL을 생성하는 AI 모델입니다.
아래는 데이터베이스 스키마(DDL)입니다.

<SCHEMA>
-- 작업완료로그 테이블
CREATE TABLE IF NOT EXISTS `log_lottranslog` (
  `TRANSLOGID` bigint(20) NOT NULL DEFAULT 0 COMMENT '로그ID',
  `LOTNO` char(20) NOT NULL DEFAULT '' COMMENT 'LOT번호',
  `LINENO` char(20) DEFAULT NULL COMMENT '생산라인번호',
  `TRANSACTIONNAME` char(50) NOT NULL DEFAULT '' COMMENT '처리명',
  `TIMELOGGED` datetime(3) NOT NULL DEFAULT '0000-00-00 00:00:00.000' COMMENT '로그입력일시',
  `ACTUALTIME` datetime(3) NOT NULL DEFAULT '0000-00-00 00:00:00.000' COMMENT '실제실행일시',
  `MATERIALCODE` char(30) NOT NULL DEFAULT '' COMMENT '자재코드',
  `MATERIALNAME` char(50) NOT NULL DEFAULT '' COMMENT '자재명',
  `TRANSQTY` double NOT NULL DEFAULT 0 COMMENT '변경수량',
  `CURRENTQTY` double NOT NULL DEFAULT 0 COMMENT '현재수량',
  `NEXTQTY` double DEFAULT NULL COMMENT '변경반영된수량',
  `TRANSUOM` char(5) DEFAULT NULL COMMENT '측정단위',
  `WAREHOUSECODE` char(20) DEFAULT NULL COMMENT '창고코드',
  `BOPMATERIALCODE` 

## 3. LoRA와 SFTConfig 설정

In [8]:
peft_config = LoraConfig(
        lora_alpha=32,
        lora_dropout=0.1,
        r=8,
        bias="none",
        target_modules=["q_proj", "v_proj"],
        task_type="CAUSAL_LM",
)

`lora_alpha`: LoRA(Low-Rank Adaptation)에서 사용하는 스케일링 계수를 설정합니다. LoRA의 가중치 업데이트가 모델에 미치는 영향을 조정하는 역할을 하며, 일반적으로 학습 안정성과 관련이 있습니다.

`lora_dropout`: LoRA 적용 시 드롭아웃 확률을 설정합니다. 드롭아웃은 과적합(overfitting)을 방지하기 위해 일부 뉴런을 랜덤하게 비활성화하는 정규화 기법입니다. `0.1`로 설정하면 학습 중 10%의 뉴런이 비활성화됩니다.

`r`: LoRA의 랭크(rank)를 설정합니다. 이는 LoRA가 학습할 저차원 공간의 크기를 결정합니다. 작은 값일수록 계산 및 메모리 효율이 높아지지만 모델의 학습 능력이 제한될 수 있습니다.

`bias`: LoRA 적용 시 편향(bias) 처리 방식을 지정합니다. `"none"`으로 설정하면 편향이 LoRA에 의해 조정되지 않습니다. `"all"` 또는 `"lora_only"`와 같은 값으로 변경하여 편향을 조정할 수도 있습니다.

`target_modules`: LoRA를 적용할 특정 모듈(레이어)의 이름을 리스트로 지정합니다. 예제에서는 `"q_proj"`와 `"v_proj"`를 지정하여, 주로 Self-Attention 메커니즘의 쿼리와 값 프로젝션 부분에 LoRA를 적용합니다.

`task_type`: LoRA가 적용되는 작업 유형을 지정합니다. `"CAUSAL_LM"`은 Causal Language Modeling, 즉 시퀀스 생성 작업에 해당합니다. 다른 예로는 `"SEQ2SEQ_LM"`(시퀀스-투-시퀀스 언어 모델링) 등이 있습니다.

In [9]:
# 최대 길이
max_seq_length=8192

args = SFTConfig(
    output_dir="llama-8b-text-to-sql-ko",           # 저장될 디렉토리와 저장소 ID
    num_train_epochs=5,                      # 학습할 총 에포크 수 
    per_device_train_batch_size=4,           # GPU당 배치 크기
    gradient_accumulation_steps=2,           # 그래디언트 누적 스텝 수
    gradient_checkpointing=True,             # 메모리 절약을 위한 체크포인팅
    optim="adamw_torch_fused",               # 최적화기
    logging_steps=10,                        # 로그 기록 주기
    save_strategy="steps",                   # 저장 전략
    save_steps=50,                           # 저장 주기
    bf16=True,                              # bfloat16 사용
    learning_rate=1e-4,                     # 학습률
    max_grad_norm=0.3,                      # 그래디언트 클리핑
    warmup_ratio=0.03,                      # 워밍업 비율
    lr_scheduler_type="constant",           # 고정 학습률
    push_to_hub=False,                      # 허브 업로드 안 함
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    report_to=None,
    max_length=max_seq_length,              # 최대 시퀀스 길이 추가
)

`output_dir`: 학습 결과가 저장될 디렉토리 또는 모델 저장소의 이름을 지정합니다. 이 디렉토리에 학습된 모델 가중치, 설정 파일, 로그 파일 등이 저장됩니다.

`num_train_epochs`: 모델을 학습시키는 총 에포크(epoch) 수를 지정합니다. 에포크는 학습 데이터 전체를 한 번 순회한 주기를 의미합니다. 예를 들어, `3`으로 설정하면 데이터셋을 3번 학습합니다.

`per_device_train_batch_size`: GPU 한 대당 사용되는 배치(batch)의 크기를 설정합니다. 배치 크기는 모델이 한 번에 처리하는 데이터 샘플의 수를 의미합니다. 작은 크기는 메모리 사용량이 적지만 학습 시간이 증가할 수 있습니다.

`gradient_accumulation_steps`: 그래디언트를 누적할 스텝(step) 수를 지정합니다. 이 값이 2로 설정된 경우, 두 스텝마다 그래디언트를 업데이트합니다. 배치 크기를 가상으로 늘리는 효과가 있으며, GPU 메모리 부족 문제를 해결할 때 유용합니다.

`gradient_checkpointing`: 그래디언트 체크포인팅을 활성화하여 메모리를 절약합니다. 이 옵션은 계산 그래프를 일부 저장하지 않고 다시 계산하여 메모리를 절약하지만, 속도가 약간 느려질 수 있습니다.

`optim`: 학습 시 사용할 최적화 알고리즘을 설정합니다. `adamw_torch_fused`는 PyTorch의 효율적인 AdamW 최적화기를 사용합니다.

`logging_steps`: 로그를 기록하는 주기를 스텝 단위로 지정합니다. 예를 들어, `10`으로 설정하면 매 10 스텝마다 로그를 기록합니다.

`save_strategy`: 모델을 저장하는 전략을 설정합니다. `"steps"`로 설정된 경우, 지정된 스텝마다 모델이 저장됩니다.

`save_steps`: 모델을 저장하는 주기를 스텝 단위로 설정합니다. 예를 들어, `50`으로 설정하면 매 50 스텝마다 모델을 저장합니다.

`bf16`: bfloat16 정밀도를 사용하도록 설정합니다. bfloat16은 FP32와 유사한 범위를 제공하면서 메모리와 계산 효율성을 높입니다.

`learning_rate`: 학습률을 지정합니다. 학습률은 모델의 가중치가 한 번의 업데이트에서 얼마나 크게 변할지를 결정합니다. 일반적으로 작은 값을 사용하여 안정적인 학습을 유도합니다.

`max_grad_norm`: 그래디언트 클리핑의 임계값을 설정합니다. 이 값보다 큰 그래디언트가 발생하면, 임계값으로 조정하여 폭발적 그래디언트를 방지합니다.

`warmup_ratio`: 학습 초기 단계에서 학습률을 선형으로 증가시키는 워밍업 비율을 지정합니다. 학습의 안정성을 높이기 위해 사용됩니다.

`lr_scheduler_type`: 학습률 스케줄러의 유형을 설정합니다. `"constant"`는 학습률을 일정하게 유지합니다.

`push_to_hub`: 학습된 모델을 허브에 업로드할지 여부를 설정합니다. `False`로 설정하면 업로드하지 않습니다.

`remove_unused_columns`: 사용되지 않는 열을 제거할지 여부를 설정합니다. True로 설정하면 메모리를 절약할 수 있습니다.

`dataset_kwargs`: 데이터셋 로딩 시 추가적인 설정을 전달합니다. 예제에서는 `skip_prepare_dataset: True`로 설정하여 데이터셋 준비 단계를 건너뜹니다.

`report_to`: 학습 로그를 보고할 대상을 지정합니다. `None`으로 설정되면 로그가 기록되지 않습니다.

## 4. 학습 중 전처리 함수: collate_fn

In [10]:
def collate_fn(batch):
    new_batch = {
        "input_ids": [],
        "attention_mask": [],
        "labels": []
    }

    for example in batch:
        messages = example["messages"]

        # LLaMA 3 채팅 템플릿 적용 (시작 토큰 포함)
        prompt = "<|begin_of_text|>"
        for msg in messages:
            role = msg["role"]
            content = msg["content"].strip()
            prompt += f"<|start_header_id|>{role}<|end_header_id|>\n{content}<|eot_id|>"

        # 마지막 assistant 메시지는 응답으로 간주하고 레이블에 포함
        text = prompt.strip()

        # 토큰화
        tokenized = tokenizer(
            text,
            truncation=True,
            max_length=max_seq_length,
            padding=False,
            return_tensors=None,
        )

        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]
        labels = [-100] * len(input_ids)

        # assistant 응답의 시작 위치 찾기
        assistant_header = "<|start_header_id|>assistant<|end_header_id|>\n"
        assistant_tokens = tokenizer.encode(assistant_header, add_special_tokens=False)
        eot_token = "<|eot_id|>"
        eot_tokens = tokenizer.encode(eot_token, add_special_tokens=False)

        # 레이블 범위 지정
        i = 0
        while i <= len(input_ids) - len(assistant_tokens):
            if input_ids[i:i + len(assistant_tokens)] == assistant_tokens:
                start = i + len(assistant_tokens)
                end = start
                while end <= len(input_ids) - len(eot_tokens):
                    if input_ids[end:end + len(eot_tokens)] == eot_tokens:
                        break
                    end += 1
                for j in range(start, end):
                    labels[j] = input_ids[j]
                for j in range(end, end + len(eot_tokens)):
                    labels[j] = input_ids[j]  # <|eot_id|> 토큰도 포함
                break
            i += 1

        new_batch["input_ids"].append(input_ids)
        new_batch["attention_mask"].append(attention_mask)
        new_batch["labels"].append(labels)

    # 패딩 처리
    max_length = max(len(ids) for ids in new_batch["input_ids"])
    for i in range(len(new_batch["input_ids"])):
        pad_len = max_length - len(new_batch["input_ids"][i])
        new_batch["input_ids"][i].extend([tokenizer.pad_token_id] * pad_len)
        new_batch["attention_mask"][i].extend([0] * pad_len)
        new_batch["labels"][i].extend([-100] * pad_len)

    for k in new_batch:
        new_batch[k] = torch.tensor(new_batch[k])

    return new_batch

`collate_fn(batch)` 함수는 자연어 처리 모델 학습을 위해 데이터를 전처리하는 역할을 수행합니다. 이 함수는 배치 내의 데이터를 처리하여 모델이 사용할 수 있는 입력 형식으로 변환합니다.

먼저, 각 샘플의 메시지에서 개행 문자를 제거하고 필요한 정보만 남깁니다. 정리된 메시지로 텍스트를 구성하고 이를 토큰화하여 input_ids와 attention_mask를 생성합니다. 이후 레이블 데이터를 초기화한 다음, 특정 토큰 패턴(<|im_start|>assistant 이후부터 <|im_end|>까지)을 찾아 해당 범위에 레이블을 설정합니다. 이 범위를 제외한 나머지 위치는 -100으로 설정하여 손실 계산에서 제외되도록 합니다.

최종적으로, 배치 내 모든 샘플의 길이를 동일하게 맞추기 위해 패딩 작업을 수행합니다. 이 과정에서 입력 데이터에는 패딩 토큰 ID를 추가하고, 어텐션 마스크에는 0을 추가하며, 레이블에는 -100을 추가합니다. 모든 데이터는 PyTorch 텐서로 변환되어 반환됩니다.

In [11]:
# collate_fn 테스트 (배치 크기 1로)
example = train_dataset[0]
batch = collate_fn([example])

print("\n처리된 배치 데이터:")
print("입력 ID 형태:", batch["input_ids"].shape)
print("어텐션 마스크 형태:", batch["attention_mask"].shape)
print("레이블 형태:", batch["labels"].shape)


처리된 배치 데이터:
입력 ID 형태: torch.Size([1, 2817])
어텐션 마스크 형태: torch.Size([1, 2817])
레이블 형태: torch.Size([1, 2817])


In [12]:
print('입력에 대한 정수 인코딩 결과:')
print(batch["input_ids"][0].tolist())

입력에 대한 정수 인코딩 결과:
[128000, 128006, 9125, 128007, 198, 65895, 83628, 34804, 8029, 18359, 53017, 44005, 15592, 122180, 80052, 627, 54059, 54542, 16969, 55348, 105010, 107128, 80307, 45780, 93220, 100711, 7, 59881, 8, 80052, 382, 27, 3624, 36939, 397, 313, 121002, 124790, 82750, 107573, 13094, 105551, 198, 23421, 14700, 11812, 4276, 35939, 1595, 848, 92949, 1485, 848, 63, 2456, 220, 1595, 44203, 7391, 926, 63, 80763, 7, 508, 8, 4276, 1808, 12221, 220, 15, 51605, 364, 82750, 926, 756, 220, 1595, 16122, 9173, 63, 1181, 7, 508, 8, 4276, 1808, 12221, 3436, 51605, 364, 16122, 73148, 756, 220, 1595, 64227, 64462, 63, 1181, 7, 508, 8, 12221, 1808, 51605, 364, 77535, 86157, 109317, 73148, 756, 220, 1595, 49873, 47259, 7687, 63, 1181, 7, 1135, 8, 4276, 1808, 12221, 3436, 51605, 364, 102657, 29102, 80732, 756, 220, 1595, 18621, 7391, 46622, 63, 9050, 7, 18, 8, 4276, 1808, 12221, 364, 931, 15, 12, 410, 12, 410, 220, 410, 25, 410, 25, 410, 13, 931, 6, 51605, 364, 82750, 44966, 29854, 33177, 30426, 75

In [14]:
# 디코딩된 input_ids 출력
decoded_text = tokenizer.decode(
    batch["input_ids"][0].tolist(),
    skip_special_tokens=False,
    clean_up_tokenization_spaces=False
)

print("\ninput_ids 디코딩 결과:")
print(decoded_text)


input_ids 디코딩 결과:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
당신은 SQL을 생성하는 AI 모델입니다.
아래는 데이터베이스 스키마(DDL)입니다.

<SCHEMA>
-- 작업완료로그 테이블
CREATE TABLE IF NOT EXISTS `log_lottranslog` (
  `TRANSLOGID` bigint(20) NOT NULL DEFAULT 0 COMMENT '로그ID',
  `LOTNO` char(20) NOT NULL DEFAULT '' COMMENT 'LOT번호',
  `LINENO` char(20) DEFAULT NULL COMMENT '생산라인번호',
  `TRANSACTIONNAME` char(50) NOT NULL DEFAULT '' COMMENT '처리명',
  `TIMELOGGED` datetime(3) NOT NULL DEFAULT '0000-00-00 00:00:00.000' COMMENT '로그입력일시',
  `ACTUALTIME` datetime(3) NOT NULL DEFAULT '0000-00-00 00:00:00.000' COMMENT '실제실행일시',
  `MATERIALCODE` char(30) NOT NULL DEFAULT '' COMMENT '자재코드',
  `MATERIALNAME` char(50) NOT NULL DEFAULT '' COMMENT '자재명',
  `TRANSQTY` double NOT NULL DEFAULT 0 COMMENT '변경수량',
  `CURRENTQTY` double NOT NULL DEFAULT 0 COMMENT '현재수량',
  `NEXTQTY` double DEFAULT NULL COMMENT '변경반영된수량',
  `TRANSUOM` char(5) DEFAULT NULL COMMENT '측정단위',
  `WAREHOUSECODE` char(20) DEFAULT NULL COMMENT '창고코드',
  

In [13]:
print('레이블에 대한 정수 인코딩 결과:')
print(batch["labels"][0].tolist())

레이블에 대한 정수 인코딩 결과:
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -1

LLM 학습에서 `input_ids`와 `labels`는 모델의 학습 목표에 따라 생성됩니다. 이를 예시 문장과 정수 인코딩을 통해 상세히 설명하겠습니다.


예를 들어, 다음과 같은 대화 데이터를 모델이 학습해야 한다고 가정합니다.  
사용자가 **"안녕하세요, 오늘 날씨는 어떤가요?"**라고 물었고,  
모델은 **"안녕하세요! 오늘 날씨는 맑고 화창합니다."**라고 응답해야 한다고 합시다.

이 데이터를 학습하기 위해 먼저 전체 대화 데이터를 정수로 인코딩합니다.  
토크나이저는 대화의 구조를 구분하기 위해 `<|im_start|>`와 `<|im_end|>`을 사용하여 정수 시퀀스를 생성한다고 가정해봅시다.  
(실제로는 LLM 템플릿이 이보다는 복잡함을 기억하고 혼동하지 맙시다.)
정수 시퀀스는 다음과 같이 구성될 수 있습니다.  

```
<|im_start|>user 안녕하세요, 오늘 날씨는 어떤가요?<|im_end|>  
<|im_start|>assistant 안녕하세요! 오늘 날씨는 맑고 화창합니다.<|im_end|>
```

이를 정수로 변환하면 다음과 같습니다.  
`input_ids = [1001, 2001, 3001, 4001, 5001, 6001, 7001, 1002, 1001, 8001, 9001, 1003, 2002]`  
모델이 예측해야 하는 부분은 `assistant`의 응답인 "안녕하세요! 오늘 날씨는 맑고 화창합니다."입니다. 따라서, `labels`는 다음과 같이 설정됩니다.

`labels = [-100, -100, -100, -100, -100, -100, -100, -100, -100, 8001, 9001, 1003, 2002]`  

이처럼 `labels`는 모델의 출력이 필요한 영역만을 포함하고, 나머지 부분은 `-100`으로 채워져 모델이 실제로 예측하고 오차를 계산해야 하는 대상(학습 대상)에서 제외됩니다. 이를 통해 모델은 불필요한 영역을 학습하지 않고, 필요한 응답 영역에만 집중할 수 있습니다.


## 5. 학습

In [17]:
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    peft_config=peft_config,
)

/usr/local/lib/python3.11/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
# 학습 시작
trainer.train()   # 모델이 자동으로 허브와 output_dir에 저장됨

# 모델 저장
trainer.save_model()   # 최종 모델을 저장

Step,Training Loss
10,1.166800
20,0.614200
30,0.401300
40,0.324600
50,0.260600
60,0.240900
70,0.201800
80,0.208200
90,0.168100
100,0.150300
